# 🚦 STC - Treinamento do YOLOv8 no Google Colab
**Sistema de Controle de Tráfego (STC)**

Este notebook treina um modelo **YOLOv8** (biblioteca Ultralytics) para detectar objetos (ex.: carros, motos, ônibus, caminhões) com o **seu dataset**.
Baseado no notebook de Evan Juras (EJ Technology Consultants), adaptado e explicado para o STC.

## O que é cada coisa?
- **YOLO** ("You Only Look Once"): família de redes neurais que detecta objetos em uma imagem em uma única passada, por isso é rápida (serve para vídeo em tempo real).
- **Ultralytics**: biblioteca Python que implementa o YOLOv8 e traz comandos prontos (`yolo detect train`, `predict`, `export`).
- **Google Colab**: computador virtual (Linux) no navegador, com **GPU gratuita**. Treinar na GPU é dezenas de vezes mais rápido que na CPU.
- **Transfer learning**: não treinamos do zero. Partimos de um modelo pré-treinado (`yolov8s.pt`, que já conhece o dataset COCO) e o ajustamos para as *suas* classes. Por isso poucas centenas de imagens bastam.

## Fluxo geral
`GPU → dataset → dividir treino/validação → instalar → data.yaml → treinar → testar → baixar → usar`

## Como usar o Colab
- Cada bloco de código tem um botão ▶ (Play). Execute **na ordem**, de cima para baixo, esperando cada um terminar.
- Linhas começando com `!` são comandos de terminal Linux executados pelo Colab. Linhas sem `!` são Python.
- Linhas começando com `#` são comentários (explicações, não executam).

## ⚠️ Antes de abrir este notebook
Este notebook cobre só a **etapa de treino (Colab)**. Antes dele você precisa, no seu PC:
1. **Instalar o Anaconda** e criar o ambiente: `conda create --name YOLO_env python=3.12 -y` e `conda activate YOLO_env`.
2. **Instalar e iniciar o Label Studio**: `pip install label-studio` e `label-studio start` (abre em http://localhost:8080).
3. **Tirar as fotos** do que quer detectar (~200 imagens, variando fundo e iluminação).
4. **Rotular as imagens** no Label Studio e **exportar em formato YOLO** como `data.zip`.

Tutorial completo com prints: **README_TREINAMENTO_YOLOV8.md** (etapas A e B).

## 0. Verificar se a GPU está ativa

**Por quê?** O treino envolve milhões de cálculos matriciais; a GPU faz isso em paralelo. Sem GPU o treino pode levar horas em vez de minutos.

**Como ativar:** menu **Ambiente de execução → Alterar tipo de ambiente de execução → Acelerador de hardware → GPU (ex.: T4) → Salvar**.

In [ ]:
# nvidia-smi = "NVIDIA System Management Interface".
# Mostra a placa de vídeo disponível, memória usada/total e processos na GPU.
# Se aparecer uma tabela com "Tesla T4" (ou similar), está tudo certo.
# Se der erro "command not found", a GPU NÃO está ativada: volte ao menu acima.
!nvidia-smi

## 1. Coletar e rotular as imagens

Antes de treinar, o modelo precisa de **exemplos**: imagens + a localização de cada objeto nelas (rótulos/anotações).

**Boas práticas**
- Ponto de partida para prova de conceito: **~200 imagens** (mais é melhor).
- Varie fundo, iluminação (dia/noite/chuva), ângulo e distância; inclua outros objetos na cena para o modelo aprender o que *não* é a classe.
- Para o STC, use imagens parecidas com as da câmera real que será usada em produção.

**Onde obter**
- Dataset próprio (melhor resultado): tire fotos/frames de vídeo e rotule com o [Label Studio](https://labelstud.io) (gratuito).
- Datasets prontos: [Roboflow Universe](https://universe.roboflow.com), [Kaggle](https://www.kaggle.com/datasets).

**Formato YOLO dos rótulos:** cada imagem `foto1.jpg` tem um `foto1.txt` com uma linha por objeto:
```
<id_da_classe> <x_centro> <y_centro> <largura> <altura>   (valores de 0 a 1, relativos ao tamanho da imagem)
```

**Estrutura obrigatória do `data.zip`:**
```
data.zip
├── images/       ← as imagens (.jpg / .png)
├── labels/       ← um .txt por imagem, no formato acima
└── classes.txt   ← nomes das classes, uma por linha (a linha 1 = id 0, linha 2 = id 1...)
```
Exemplo de `classes.txt`:
```
carro
moto
onibus
caminhao
```
> O Label Studio exporta isso tudo em um `project.zip` (também gera `notes.json`, que pode ser ignorado). Renomeie para `data.zip`.

## 1.1 Rotular com o Label Studio (passo a passo)
Depois de tirar as fotos do que você quer detectar, rotule-as **no seu PC** (não no Colab):
1. Instale e abra: `pip install label-studio` → `label-studio start` → acesse http://localhost:8080.
2. Na tela inicial clique em **Create Project** e dê um nome.
3. Aba **Data Import**: faça upload das imagens (arraste ou **Upload Files**).
4. Aba **Labeling Setup**: escolha o template **Object Detection with Bounding Boxes**, apague as labels de exemplo e adicione as suas classes (uma por linha, ex.: carro, moto, onibus, caminhao). A ordem define o id (0, 1, 2...). Clique em **Save**.
5. Abra o projeto → **Label All Tasks**: selecione a label, arraste uma caixa sobre cada objeto e clique em **Submit**. Repita em todas as imagens.
6. **Export** → formato **YOLO** → renomeie o zip para `data.zip`.

📷 Prints de cada etapa: veja o arquivo README_TREINAMENTO_YOLOV8.md (seção 1.2) no repositório.

## 2. Enviar o dataset para o Colab

O Colab roda em outro computador (na nuvem), então ele **não enxerga os arquivos do seu PC**. É preciso enviar o `data.zip`. Escolha **uma** das opções (A, B ou C).

> ⚠️ Os arquivos do Colab são **temporários**: ao fechar/reiniciar a sessão, tudo em `/content` é apagado. Por isso baixe o modelo no final (seção 7).

### Opção A – Upload direto (simples, para datasets pequenos)
Clique no ícone de **pasta 📁** na barra lateral esquerda → ícone de upload ⬆ → selecione `data.zip`. Ele aparecerá em `/content`. Não precisa executar código.

### Opção B – Google Drive (recomendado se >50MB)
Suba o `data.zip` no seu Drive uma vez e reaproveite sempre que reiniciar o Colab. Ajuste o caminho no código abaixo.

In [ ]:
# OPÇÃO B: copiar o data.zip do Google Drive para o Colab.

# Importa o módulo do Colab que sabe se conectar ao Google Drive.
from google.colab import drive

# "Monta" o seu Drive como se fosse uma pasta em /content/gdrive.
# Vai abrir uma janela pedindo autorização para acessar o Drive.
drive.mount('/content/gdrive')

# cp = copiar. Copia o zip do Drive para /content (disco local do Colab, bem mais rápido de ler).
# >>> TROQUE "STC/data.zip" pelo caminho real do seu arquivo dentro do "Meu Drive".
!cp /content/gdrive/MyDrive/STC/data.zip /content

### Opção C – Dataset de exemplo (só para testar o fluxo)
Se quiser apenas ver o notebook funcionando antes de ter o seu dataset, descomente a linha abaixo (remova o `#`). É o dataset de doces do autor original.

In [ ]:
# wget = baixa um arquivo da internet.
#   -O /content/data.zip  → salva com este nome/local.
# (descomente a linha abaixo para usar; deixe comentada se usou a opção A ou B)
# !wget -O /content/data.zip https://s3.us-west-1.amazonaws.com/evanjuras.com/resources/candy_data_06JAN25.zip

## 3. Descompactar e dividir em treino / validação

Depois de enviado, o `data.zip` deve aparecer na pasta 📁 à esquerda.

### 3.1 Descompactar

In [ ]:
# unzip = descompacta o arquivo.
#   -q                        → modo silencioso (não lista cada arquivo).
#   /content/data.zip         → arquivo de origem.
#   -d /content/custom_data   → pasta de destino (será criada).
# Resultado: /content/custom_data/images, /labels e classes.txt
!unzip -q /content/data.zip -d /content/custom_data

### 3.2 Dividir em treino e validação

O Ultralytics exige que o dataset esteja dividido em dois grupos:

| Pasta | Para que serve |
|-------|----------------|
| **train** (90%) | Imagens que a rede *usa para aprender*. Em cada época ela vê todas e ajusta seus pesos. |
| **validation** (10%) | Imagens que a rede *nunca usa para aprender*; servem para medir, a cada época, se ela está realmente generalizando ou só "decorando" o treino. |

Cada grupo precisa ter as subpastas `images/` e `labels/`. O script do autor cria essa estrutura (`/content/data/train/...` e `/content/data/validation/...`) e sorteia aleatoriamente qual imagem vai para onde.

In [ ]:
# Baixa o script de divisão do GitHub do autor para /content/train_val_split.py.
!wget -O /content/train_val_split.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/utils/train_val_split.py

# Executa o script:
#   --datapath  → pasta onde estão images/, labels/ e classes.txt (a que descompactamos).
#   --train_pct → fração para treino. 0.9 = 90% treino / 10% validação.
!python train_val_split.py --datapath="/content/custom_data" --train_pct=0.9

## 4. Instalar o Ultralytics

Instala a biblioteca que contém o YOLOv8 e o comando `yolo`. Junto vêm dependências como **PyTorch** (framework de redes neurais), **OpenCV** (imagens/vídeo) e **NumPy**. Leva ~1 minuto.

In [ ]:
# pip = gerenciador de pacotes do Python. Baixa e instala o ultralytics e suas dependências.
!pip install ultralytics

## 5. Criar o arquivo de configuração `data.yaml`

O treino precisa saber **onde estão** as imagens de treino/validação e **quais são as classes**. Isso é descrito num arquivo `data.yaml`, com os campos:

| Campo | Significado |
|-------|-------------|
| `path` | pasta raiz do dataset já dividido (`/content/data`) |
| `train` | subpasta com imagens de treino (relativa a `path`) |
| `val` | subpasta com imagens de validação |
| `nc` | número de classes |
| `names` | lista com o nome de cada classe, na ordem dos ids |

Em vez de escrever à mão, o código abaixo lê o seu `classes.txt` e gera o arquivo.

In [ ]:
import yaml   # biblioteca para ler/escrever arquivos .yaml
import os      # funções do sistema de arquivos (ex.: checar se um arquivo existe)

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
    # 1) Confere se o classes.txt existe; se não, avisa e interrompe.
    if not os.path.exists(path_to_classes_txt):
        print(f'classes.txt não encontrado! Crie o arquivo em {path_to_classes_txt}')
        return

    # 2) Lê o classes.txt: cada linha não vazia vira o nome de uma classe.
    with open(path_to_classes_txt, 'r') as f:
        classes = [line.strip() for line in f.readlines() if len(line.strip()) > 0]

    # 3) Monta o dicionário com a configuração.
    data = {
        'path': '/content/data',        # raiz criada pelo script de divisão
        'train': 'train/images',        # imagens de treino
        'val': 'validation/images',     # imagens de validação
        'nc': len(classes),             # quantidade de classes
        'names': classes                # nomes das classes
    }

    # 4) Grava em formato YAML (sort_keys=False mantém a ordem acima).
    with open(path_to_data_yaml, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
    print(f'Arquivo criado em {path_to_data_yaml}')

# Executa a função com os caminhos do nosso projeto.
create_data_yaml('/content/custom_data/classes.txt', '/content/data.yaml')

# cat = mostra o conteúdo do arquivo, para você conferir se as classes estão corretas.
print('\nConteúdo do data.yaml:\n')
!cat /content/data.yaml

## 6. Treinar o modelo 🏋️

### 6.1 Parâmetros importantes

**`model` – tamanho do modelo**

| Modelo | Velocidade | Precisão | Quando usar |
|--------|-----------|----------|-------------|
| `yolov8n.pt` (nano) | ⚡⚡⚡ | + | dispositivos fracos (Raspberry Pi) |
| `yolov8s.pt` (small) | ⚡⚡ | ++ | **bom ponto de partida** |
| `yolov8m.pt` (medium) | ⚡ | +++ | PC com GPU |
| `yolov8l.pt` / `yolov8x.pt` | 🐢 | ++++ | máxima precisão, mais lento |

**`epochs` – quantas vezes a rede vê o dataset inteiro.** Poucas épocas = modelo mal treinado (*underfitting*); muitas = decora o treino (*overfitting*). Sugestão: ~60 para menos de 200 imagens; ~40 para mais de 200.

**`imgsz` – resolução das imagens no treino.** Padrão 640 (640×640 px). Resolução menor (ex.: 480) = mais rápido e menos preciso; objetos pequenos pedem resolução maior.

### 6.2 Executar o treino

In [ ]:
# yolo detect train → comando da Ultralytics para treinar um detector de objetos.
#   data=/content/data.yaml  → arquivo de configuração criado no passo 5.
#   model=yolov8s.pt         → ponto de partida pré-treinado (baixado automaticamente).
#   epochs=60                → número de épocas.
#   imgsz=640                → resolução de treino.
#
# A cada época o programa:
#   1. treina com as imagens de "train";
#   2. valida com as de "validation" e imprime Precision, Recall e mAP.
# O mAP (precisão média) deve subir ao longo das épocas; quanto mais perto de 1.0, melhor.
#
# ⚠️ Deixe o treino ir até o FIM: no final ele roda um otimizador que remove partes
# desnecessárias do modelo (deixa o arquivo menor e pronto para uso).
!yolo detect train data=/content/data.yaml model=yolov8s.pt epochs=60 imgsz=640

### 6.3 Onde ficam os resultados
Em `/content/runs/detect/train/`:
- `weights/best.pt` → **melhor modelo** (maior mAP na validação). É este que você usa.
- `weights/last.pt` → modelo da última época.
- `results.png` → gráficos de loss, precisão, recall e mAP por época.
- `confusion_matrix.png` → mostra quais classes o modelo confunde entre si.

**Como interpretar:** as curvas de *loss* devem cair; as de *mAP* devem subir e estabilizar. Se a loss de treino cai mas a de validação sobe, há overfitting (use mais imagens ou menos épocas).

In [ ]:
# (Opcional) Exibe o gráfico de resultados do treino direto no notebook.
from IPython.display import Image, display
display(Image(filename='/content/runs/detect/train/results.png', width=900))

## 7. Testar o modelo ✅

Agora verificamos, com imagens reais, se o modelo detecta o que deveria. Usamos as imagens de **validação** (que não foram usadas para ajustar os pesos).

In [ ]:
# yolo detect predict → roda o modelo em imagens/vídeos (inferência).
#   model=.../best.pt                 → modelo treinado que acabamos de gerar.
#   source=data/validation/images     → pasta de imagens (também aceita vídeo .mp4 ou imagem única).
#   save=True                         → salva as imagens com as caixas desenhadas
#                                       em /content/runs/detect/predict/.
!yolo detect predict model=runs/detect/train/weights/best.pt source=data/validation/images save=True

In [ ]:
import glob
from IPython.display import Image, display

# Lista as imagens geradas pelo predict e mostra as 10 primeiras (altura 400px).
# Cada objeto detectado deve ter uma caixa com o nome da classe e a confiança (0 a 1).
for image_path in glob.glob('/content/runs/detect/predict/*.jpg')[:10]:
    display(Image(filename=image_path, height=400))
    print('\n')

**O modelo está detectando mal? Dicas:**
1. Confira se há erros de rotulagem (caixa errada, classe trocada, objetos sem rótulo).
2. Aumente `epochs`.
3. Use um modelo maior (ex.: `yolov8m.pt` ou `yolov8l.pt`).
4. Adicione mais imagens variadas ao dataset.

**Testar em vídeo:** envie um vídeo para o Colab e use `source=/content/meu_video.mp4` no comando acima. O resultado fica em `runs/detect/predict`.

## 8. Baixar o modelo treinado 💾

Lembre: tudo em `/content` some quando a sessão acabar. Vamos empacotar o modelo e baixar para o seu PC.

In [ ]:
# mkdir -p = cria a pasta (sem erro se já existir) que vai guardar o modelo e os resultados.
!mkdir -p /content/my_model

# Copia o melhor modelo e o renomeia para my_model.pt.
!cp /content/runs/detect/train/weights/best.pt /content/my_model/my_model.pt

# Copia (-r = recursivo) toda a pasta de resultados do treino, para consulta futura.
!cp -r /content/runs/detect/train /content/my_model

# %cd = entra na pasta (comando mágico do Jupyter, mantém a mudança entre células).
%cd my_model

# zip = compacta. Primeiro o modelo, depois (-r) a pasta "train" com os resultados.
!zip /content/my_model.zip my_model.pt
!zip -r /content/my_model.zip train

# Volta para a pasta /content.
%cd /content

In [ ]:
# Dispara o download do my_model.zip pelo navegador.
# Pode demorar bastante; se travar, baixe pelo painel de arquivos 📁 (botão direito → Fazer download).
from google.colab import files
files.download('/content/my_model.zip')

### 8.1 (Opcional) Exportar para ONNX / TFLite
O `.pt` roda com Python + PyTorch. Para outros ambientes (C++, celular, Raspberry Pi) converta o modelo:
- **ONNX**: formato universal, roda em vários runtimes.
- **TFLite**: otimizado para celulares e dispositivos de borda.

In [ ]:
# yolo export → converte o modelo para outro formato. O arquivo é criado ao lado do best.pt.
#   format=onnx    → ONNX. Troque por format=tflite para TensorFlow Lite.
!yolo export model=/content/runs/detect/train/weights/best.pt format=onnx

## 9. Rodar o modelo no seu PC 🖥️

Faça no **seu computador** (não no Colab). Descompacte o `my_model.zip` baixado e, num terminal:

```bash
# 1) Cria um ambiente Python isolado (evita conflito com outros projetos) e ativa.
conda create --name yolo-env1 python=3.12 -y
conda activate yolo-env1

# 2) Instala o Ultralytics (traz PyTorch, OpenCV, NumPy).
pip install ultralytics

# 3) (Opcional, só com GPU NVIDIA) instala PyTorch com suporte a CUDA para acelerar.
pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

# 4) Entra na pasta onde está o my_model.pt.
cd caminho/para/my_model

# 5) Baixa o script de exemplo que carrega o modelo e desenha as caixas.
curl -o yolo_detect.py https://raw.githubusercontent.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/refs/heads/main/yolo_detect.py

# 6) Executa na webcam USB (usb0) a 1280x720. Também aceita vídeo, imagem ou pasta.
python yolo_detect.py --model my_model.pt --source usb0 --resolution 1280x720
```
Use `python yolo_detect.py --help` para ver todas as opções.

## 10. Erros comuns
- **Erro de pasta/arquivo não encontrado** → confira a estrutura do dataset (`images/`, `labels/`, `classes.txt`).
- **Classes erradas ou erro no `data.yaml`** → confira typos e linhas em branco no `classes.txt`.
- **`nvidia-smi` falha** → ative a GPU (seção 0).
- **Sessão caiu / arquivos sumiram** → o Colab é temporário; use o Drive (opção B) e baixe o modelo ao final.
- Persistindo, pesquise a mensagem de erro ou abra uma [Issue no GitHub](https://github.com/EdjeElectronics/Train-and-Deploy-YOLO-Models/issues).

**Parabéns, seu modelo do STC foi treinado!** 🚦 Próximo passo: integrá-lo ao sistema (contagem de veículos, registro ao longo do tempo, etc.).